# Phase 1: Exploring the cleaned tables with SQL

This notebook connects to the database that `scripts/run_sql.py` built and lets you practice SQL on the real Freddie Mac data. Run `python scripts/run_sql.py` first.

- **The two tables:** `loans` (one row per loan) and `loan_months` (one row per loan per month).
- **How the exercises work:** each one has a SQL query with `...` blanks to fill in, then a check cell. The checks test the shape of your answer (column names, number of rows, sensible ranges) without giving the answer away. Solutions are in `01_explore_phase1_solutions.ipynb`.
- **Two rules:**
    - Before running `scripts/run_sql.py` again, run the last cell here (`con.close()`) or restart this notebook's kernel. DuckDB won't let the script write to the database while this notebook has it open.
    - Before committing this notebook to GitHub, click **Clear All Outputs**, so no loan-level rows get published.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/credit_risk.duckdb", read_only=True)   # read-only: you can't change anything by accident


def q(sql):
    # Run one SQL query and return the result as a pandas DataFrame
    return con.sql(sql).df()


def check(condition, hint):
    # Prints ✓ Looks right, or stops with a hint about what to fix
    if condition:
        print("✓ Looks right")
    else:
        raise AssertionError("Not yet: " + hint)


n_total = int(q("SELECT COUNT(*) AS n FROM loans")["n"].iloc[0])
vintages = q("SELECT DISTINCT vintage FROM loans ORDER BY vintage")["vintage"].tolist()
n_vintages = len(vintages)   # the checks below expect one row per vintage that is actually loaded
print(f"Connected: {n_total:,} loans from {n_vintages} vintages: {vintages}")
if [str(v) for v in vintages] != ["2006", "2007", "2012", "2019"]:
    print("Note: Phase 1 expects 2006, 2007, 2012, and 2019. To add a missing year, put both of its files in data/raw,")
    print("run con.close(), run python scripts/run_sql.py again, then restart this kernel. The exercises still work meanwhile.")

## Look before you query

Get to know the tables first. Run each cell and read the output.

In [ ]:
# Column names and types of the loans table
q("DESCRIBE loans")

In [ ]:
# Loans per vintage
q('''
SELECT l.vintage,
       COUNT(*) AS n_loans
FROM loans AS l
GROUP BY l.vintage
ORDER BY l.vintage
''')

In [ ]:
# A few loans (loan-level rows: clear this output before committing)
q("SELECT loan_id, vintage, fico, ltv, dti, orig_rate, state, purpose FROM loans LIMIT 5")

In [ ]:
# One loan's month-by-month history (also loan-level: clear before committing)
q('''
SELECT period, age, current_upb, dpd_status, dpd, zb_code
FROM loan_months
WHERE loan_id = (SELECT MIN(loan_id) FROM loans)
ORDER BY period
''').head(24)

## Exercises

Write each query inside the triple quotes. Run the query cell, look at the result, then run the check cell.

### 1. Average credit score and interest rate by vintage

One row per vintage with the columns `vintage`, `avg_fico`, and `avg_rate`, each average rounded to 2 decimals.

*Uses:* `AVG`, `ROUND`, `GROUP BY`, column names with `AS`.

In [ ]:
ex1 = q("""
SELECT
    vintage,
    ROUND(AVG(fico), 2) AS avg_fico,
    ROUND(AVG(orig_rate), 2) AS avg_rate
FROM loans
GROUP BY vintage
ORDER BY vintage
""")
ex1

In [ ]:
# Check
check(list(ex1.columns) == ["vintage", "avg_fico", "avg_rate"], "name the columns vintage, avg_fico, avg_rate with AS")
check(len(ex1) == n_vintages, f"one row per vintage ({n_vintages} are loaded): GROUP BY vintage, and nothing else")
check(ex1["avg_fico"].between(600, 850).all(), "average credit scores should fall between 600 and 850: AVG(fico)")
check(ex1["avg_rate"].between(1, 12).all(), "average rates should look like percentages, e.g. 6.4: AVG(orig_rate)")
check(((ex1[["avg_fico", "avg_rate"]] * 100).round() - ex1[["avg_fico", "avg_rate"]] * 100).abs().max().max() < 1e-6,
      "round to 2 decimals: ROUND(AVG(...), 2)")

### 2. Share of loans with a credit score under 620

One row per vintage with the columns `vintage` and `share_under_620`: the fraction (between 0 and 1) of loans **with a known credit score** whose score is below 620.

*Uses:* `WHERE fico IS NOT NULL`, and `AVG(CASE WHEN ... THEN 1.0 ELSE 0.0 END)`. The average of 1s and 0s is the share of 1s.

In [ ]:
ex2 = q("""
SELECT
    vintage,
    AVG(CASE WHEN fico < 620 THEN 1.0 ELSE 0.0 END) AS share_under_620
FROM loans
WHERE fico IS NOT NULL
GROUP BY vintage
ORDER BY vintage
""")
ex2

In [ ]:
# Check
check(list(ex2.columns) == ["vintage", "share_under_620"], "columns vintage and share_under_620")
check(len(ex2) == n_vintages, f"one row per vintage ({n_vintages} are loaded): GROUP BY vintage")
check(ex2["share_under_620"].between(0, 1).all(), "a share is between 0 and 1: CASE WHEN fico < 620 THEN 1.0 ELSE 0.0 END")
ref = q("SELECT vintage, COUNT(*) FILTER (WHERE fico < 620) * 1.0 / COUNT(*) AS s FROM loans WHERE fico IS NOT NULL GROUP BY vintage ORDER BY vintage")
check(((ex2["share_under_620"] - ref["s"]).abs() < 1e-9).all(), "count only loans with a known score: WHERE fico IS NOT NULL")

### 3. The 5 states with the most 2007 loans

Five rows with the columns `state` and `n_loans`, largest first, counting only the 2007 vintage.

*Uses:* `WHERE`, `GROUP BY`, `ORDER BY ... DESC`, `LIMIT`.

In [ ]:
ex3 = q("""
SELECT
    state,
    COUNT(*) AS n_loans
FROM loans
WHERE vintage = 2007
GROUP BY state
ORDER BY n_loans DESC
LIMIT 5
""")
ex3

In [ ]:
# Check
check(list(ex3.columns) == ["state", "n_loans"], "columns state and n_loans")
check(len(ex3) == 5, "LIMIT 5")
check(ex3["n_loans"].is_monotonic_decreasing, "largest first: ORDER BY n_loans DESC")
top = int(q("SELECT COUNT(*) AS n FROM loans WHERE vintage = 2007 GROUP BY state ORDER BY n DESC LIMIT 1")["n"].iloc[0])
check(int(ex3["n_loans"].iloc[0]) == top, "count only 2007 loans: WHERE vintage = 2007")

### 4. Loans per vintage by purpose

One row per (vintage, purpose) pair with the columns `vintage`, `purpose`, and `n_loans`, ordered by vintage and then by count, largest first. Purpose codes: P = purchase, C = cash-out refinance, N = no-cash-out refinance, R = refinance (not specified); blank = missing.

*Uses:* `GROUP BY` two columns, `ORDER BY` two columns.

In [ ]:
ex4 = q("""
SELECT
    vintage,
    purpose,
    COUNT(*) AS n_loans
FROM loans
GROUP BY vintage, purpose
ORDER BY vintage, n_loans DESC
""")
ex4

In [ ]:
# Check
check(list(ex4.columns) == ["vintage", "purpose", "n_loans"], "columns vintage, purpose, n_loans")
check(int(ex4["n_loans"].sum()) == n_total, "every loan should be counted once: GROUP BY vintage, purpose")
check(ex4.groupby("vintage").size().min() >= 2, "each vintage should have several purposes: group by both columns")
check(all(g["n_loans"].is_monotonic_decreasing for _, g in ex4.groupby("vintage", sort=False)),
      "within each vintage, largest count first: ORDER BY vintage, n_loans DESC")

### 5. Loans that ever reached 90+ days late

One row per vintage with the columns `vintage` and `n_ever_90`: how many different loans were 90 or more days late (`dpd >= 3`) in at least one month. Then add a third column, `share_ever_90`: that count divided by the number of loans in the vintage (use `n_ever_90 * 1.0 / 50000`, or better, a join to `loans`).

*Uses:* `COUNT(DISTINCT ...)`, `WHERE`, `GROUP BY`. `loan_months` has its own `vintage` column, so the first two columns need no join. This previews Phase 3.

In [ ]:
ex5 = q("""
SELECT
    vintage,
    COUNT(DISTINCT loan_id) AS n_ever_90,
    ((n_ever_90 * 1.0)/ 50000)   AS share_ever_90
FROM loan_months
WHERE dpd >= 3
GROUP BY vintage
ORDER BY vintage
""")
ex5

In [ ]:
# Check
check(list(ex5.columns) == ["vintage", "n_ever_90", "share_ever_90"], "columns vintage, n_ever_90, share_ever_90")
check(len(ex5) == n_vintages, f"one row per vintage ({n_vintages} are loaded): GROUP BY vintage")
ref = q("SELECT vintage, COUNT(DISTINCT loan_id) AS n FROM loan_months WHERE dpd >= 3 GROUP BY vintage ORDER BY vintage")
check((ex5["n_ever_90"].to_numpy() == ref["n"].to_numpy()).all(),
      "count each loan once (COUNT(DISTINCT loan_id)), keeping only months with dpd >= 3")
check(ex5["share_ever_90"].between(0, 1).all(), "the share is the count divided by the loans in that vintage")

## Your data-quality results

The checks from `sql/03_validate.sql`. Copy anything surprising into a message to Claude.

In [ ]:
q("SELECT check_id, check_name, vintage, result, expected, passed FROM data_quality ORDER BY check_id, vintage")

In [ ]:
# Run this before re-running scripts/run_sql.py, so the script can write to the database
con.close()